# Pipeline SARIMAX

### Informações do Grupo

##### Grupo: 4

Base: `jena_climate_2009_2016.csv`

Previsão da temperatura do ar na base Jena Climate.

- Granularidade: `horária` (agregação da base original de 10 minutos)
- Variável-alvo: `T (degC)`
- Random State (seed): `42`
- Treino-teste: `80% / 20%`

## Importações e Configurações Padrão

In [ ]:
from pathlib import Path
import itertools
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX



import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")
sys.path.insert(0, str(PROJECT_ROOT))

from validacao_bases import relatorios_como_dataframe, validar_base
from funcoes_series_temporais import (
    analisar_componente_residual,
    calcular_forca_sazonalidade,
    calcular_importancia_features,
)

DATA_FILENAME = "jena_climate_2009_2016.csv"
DATA_PATH = PROJECT_ROOT / "grupo4" / DATA_FILENAME
DATE_COLUMN = "Date Time"
TARGET_COLUMN = "T (degC)"
FREQUENCY = "1h"
TRAIN_RATIO = 0.80
SEED = 42
HORIZON = 1
SEASONAL_PERIOD = 24
FFILL_LIMIT_HOURS = 6
BASE_EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd (deg)"]
MODEL_EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd_sin", "wd_cos"]


In [ ]:
# Configuração SARIMAX para previsão móvel de uma hora.
ORDER = (2, 0, 2)
SEASONAL_ORDER = (1, 0, 1, SEASONAL_PERIOD)
TREND = "c"
EXOG_COLUMNS = MODEL_EXOG_COLUMNS
EXOG_LAG = 1
ENFORCE_STATIONARITY = False
ENFORCE_INVERTIBILITY = False
VALIDATION_HOURS = 24 * 28


## Configuração do SARIMAX

In [ ]:
# A configuração do grupo 4 está definida na seção anterior.
# Esta célula é mantida para preservar a estrutura do template.
print("Configuração do grupo 4 carregada.")

# 1. Preparação da base

## T01 — Sanitização e congelamento da base

Congelamento e validação sanitária da base: verificação de nulos, duplicatas, datas, ordenação e frequência regular.

In [ ]:
relatorio = validar_base("clima", raiz=PROJECT_ROOT)
display(relatorios_como_dataframe({"clima": relatorio}))

# T01 — limpeza e agregação horária padronizadas no grupo 4.
df_bruto = pd.read_csv(DATA_PATH)
df_bruto[DATE_COLUMN] = pd.to_datetime(
    df_bruto[DATE_COLUMN], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
n_datas_invalidas = int(df_bruto[DATE_COLUMN].isna().sum())
n_duplicadas = int(df_bruto[DATE_COLUMN].duplicated().sum())
df_bruto = (
    df_bruto.dropna(subset=[DATE_COLUMN])
    .sort_values(DATE_COLUMN)
    .drop_duplicates(subset=[DATE_COLUMN], keep="first")
    .set_index(DATE_COLUMN)
)

# -9999 é ausência, portanto deve ser removido antes da média horária.
df_numerico = df_bruto.select_dtypes(include="number").replace(-9999, np.nan)
df = df_numerico.resample(FREQUENCY).mean()

# O alvo nunca é imputado. Somente exógenas recebem preenchimento causal.
alvo_ausente = df[TARGET_COLUMN].isna().copy()
for coluna in BASE_EXOG_COLUMNS:
    df[coluna] = df[coluna].ffill(limit=FFILL_LIMIT_HOURS)
df.loc[alvo_ausente, TARGET_COLUMN] = np.nan

# Direção é circular: 0 e 360 graus representam a mesma direção.
df["wd_sin"] = np.sin(np.deg2rad(df["wd (deg)"]))
df["wd_cos"] = np.cos(np.deg2rad(df["wd (deg)"]))

if not df.index.is_monotonic_increasing or not df.index.is_unique:
    raise ValueError("O índice horário deve ser crescente e único.")
if (df["wv (m/s)"].dropna() < 0).any():
    raise ValueError("Há velocidade do vento negativa depois da limpeza.")

# O corte é definido na série horária completa, antes dos lags dos modelos.
posicao_corte = int(len(df) * TRAIN_RATIO)
FIM_TREINO = df.index[posicao_corte - 1]
INICIO_TESTE = df.index[posicao_corte]
diagnostico_base = pd.DataFrame([{
    "linhas_horarias": len(df),
    "datas_invalidas": n_datas_invalidas,
    "duplicatas_removidas": n_duplicadas,
    "horas_sem_alvo": int(df[TARGET_COLUMN].isna().sum()),
    "frequencia": FREQUENCY,
    "ffill_limite_horas": FFILL_LIMIT_HOURS,
    "fim_treino": FIM_TREINO,
    "inicio_teste": INICIO_TESTE,
}])
display(diagnostico_base)
print(f"Período: {df.index.min()} a {df.index.max()}")
print("O alvo não foi imputado; exógenas usam ffill causal limitado.")


## T02 — Dicionário de dados e disponibilidade temporal

Criação do dicionário de dados e análise de disponibilidade temporal das variáveis exógenas para prevenir data leakage.

In [ ]:
# Dicionário simplificado e disponibilidade temporal
DICIONARIO = pd.DataFrame([
    {"variavel": coluna, "tipo": str(df[coluna].dtype),
     "disponibilidade": "não conhecida no futuro; usada com defasagem de 1 hora",
     "min": df[coluna].min(), "max": df[coluna].max(), "media": df[coluna].mean()}
    for coluna in [TARGET_COLUMN] + EXOG_COLUMNS
])
display(DICIONARIO.round(3))

## T03 — Análise exploratória visual

Análise visual do alvo e das variáveis exógenas, diagnóstico de outliers e registro das decisões de limpeza.

In [ ]:
# T03 — análise exploratória visual
fig, axes = plt.subplots(
    len(EXOG_COLUMNS) + 1,
    1,
    figsize=(14, 10),
    sharex=True,
    squeeze=False,
)
axes = axes[:, 0]

axes[0].plot(
    df.index,
    df[TARGET_COLUMN],
    color="#1a3a6e",
    label="Temperatura observada",
)
axes[0].set_title("Grupo 4 — série-alvo e variáveis exógenas")
axes[0].set_ylabel("Temperatura (°C)")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

for ax, coluna in zip(axes[1:], EXOG_COLUMNS):
    ax.plot(
        df.index,
        df[coluna],
        linewidth=0.7,
        color="#e25c3e",
        label=coluna,
    )
    ax.set_title(f"Série temporal — {coluna}")
    ax.set_ylabel(coluna)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Ano")
fig.suptitle("Análise exploratória — Jena Climate", fontsize=14)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# 2. Sazonalidade

## T04 — Decomposição STL

Decomposição STL e interpretação gráfica da tendência, incluindo períodos de alta, queda e estabilidade.

In [ ]:
# Decomposição STL para a sazonalidade diária
stl_resultado = calcular_forca_sazonalidade( df[TARGET_COLUMN], periodo=SEASONAL_PERIOD)
decomposicao = stl_resultado["decomposicao"]
print(f"Força da sazonalidade: {stl_resultado['forca_sazonalidade']:.4f}")
print(f"Força da tendência: {stl_resultado['forca_tendencia']:.4f}")

fig = decomposicao.plot()
fig.set_size_inches(14, 8)
fig.suptitle("Decomposição STL — temperatura do ar", fontsize=14, y=1.02)
fig.supxlabel("Data e hora")
fig.axes[0].set_ylabel("Observado (°C)")
fig.axes[1].set_ylabel("Tendência (°C)")
fig.axes[2].set_ylabel("Sazonalidade (°C)")
fig.axes[3].set_ylabel("Resíduo (°C)")
for ax in fig.axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## T05 — Força da sazonalidade e resíduos

Cálculo da força da sazonalidade e análise do componente residual.

In [ ]:
# Análise do componente residual
analise_residuos_stl = analisar_componente_residual(decomposicao, lags=24)
print(f"Média do residual: {analise_residuos_stl['media']:.6f}")
print(f"Desvio-padrão do residual: {analise_residuos_stl['desvio_padrao']:.6f}")
display(analise_residuos_stl["autocorrelacao_residual"])

fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(
    analise_residuos_stl["residuo"],
    lags=48,
    ax=ax,
    title="ACF dos resíduos da decomposição STL",
)
ax.set_xlabel("Defasagem (lag)")
ax.set_ylabel("Autocorrelação")
ax.legend(["Resíduo STL"], loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Validação temporal

## T08 — Validação walk-forward

Implementação do protocolo walk-forward com janelas, origens de teste fixas e horizonte de previsão unificado.

In [ ]:
def construir_features_temporais(
    dados, alvo, *, lags_alvo, lags_exogenas, janelas_moveis
):
    """Cria features causais; o índice da linha representa a hora prevista."""
    features = pd.DataFrame(index=dados.index)
    for lag in sorted(set(lags_alvo)):
        if lag < HORIZON:
            raise ValueError(f"Lag do alvo deve ser >= {HORIZON}: {lag}")
        features[f"target_lag_{lag}"] = dados[alvo].shift(lag)
    for coluna, lags in lags_exogenas.items():
        for lag in sorted(set(lags)):
            if lag < HORIZON:
                raise ValueError(f"Lag da exógena deve ser >= {HORIZON}: {lag}")
            features[f"{coluna}_lag_{lag}"] = dados[coluna].shift(lag)
    passado = dados[alvo].shift(HORIZON)
    for janela in sorted(set(janelas_moveis)):
        features[f"target_roll_mean_{janela}"] = passado.rolling(janela).mean()
        features[f"target_roll_std_{janela}"] = passado.rolling(janela).std()
    features["hora_sin"] = np.sin(2 * np.pi * dados.index.hour / 24)
    features["hora_cos"] = np.cos(2 * np.pi * dados.index.hour / 24)
    features["dia_semana_sin"] = np.sin(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_semana_cos"] = np.cos(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_ano_sin"] = np.sin(2 * np.pi * dados.index.dayofyear / 365.25)
    features["dia_ano_cos"] = np.cos(2 * np.pi * dados.index.dayofyear / 365.25)
    return pd.concat([dados[alvo].rename("target"), features], axis=1).dropna()


# T08 — exógenas observadas em t-1 e calendário conhecido antecipadamente.
features = pd.DataFrame(index=df.index)
for coluna in EXOG_COLUMNS:
    features[f"{coluna}_lag1"] = df[coluna].shift(EXOG_LAG)
features["hora_sin"] = np.sin(2 * np.pi * df.index.hour / 24)
features["hora_cos"] = np.cos(2 * np.pi * df.index.hour / 24)
features["dia_ano_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
features["dia_ano_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)

# A máscara abaixo mantém os mesmos timestamps avaliados na MLP e na RF.
elegibilidade = construir_features_temporais(
    df, TARGET_COLUMN,
    lags_alvo=[1, 2, 3, 6, 12, 24, 48, 168],
    lags_exogenas={coluna: [1, 24] for coluna in MODEL_EXOG_COLUMNS},
    janelas_moveis=[6, 24, 168],
).index
model_data = pd.concat([df[TARGET_COLUMN].rename("target"), features], axis=1).dropna()
model_data = model_data.loc[model_data.index.intersection(elegibilidade)]

train = model_data.loc[model_data.index < INICIO_TESTE]
test = model_data.loc[model_data.index >= INICIO_TESTE]
if train.empty or test.empty or train.index.max() >= test.index.min():
    raise ValueError("Divisão temporal inválida.")
y_train, y_test = train["target"], test["target"]
X_train, X_test = train.drop(columns="target"), test.drop(columns="target")

print(f"Treino: {len(train):,} | teste: {len(test):,} | horizonte: {HORIZON} hora")
print(f"Corte comum: {FIM_TREINO} -> {INICIO_TESTE}")


# 4. Modelagem SARIMAX

## T10 — SARIMAX

Adaptação do SARIMAX com seleção de p, d, q, P, D, Q e m por AIC/BIC, inclusão segura de exógenas e execução walk-forward.

In [ ]:
# Grid selecionado por previsão móvel de uma hora no fim do treino.
p_range, q_range = [0, 1, 2], [0, 1, 2]
P_range, Q_range = [0, 1], [0, 1]
d_range, D_range = [0, 1], [0, 1]
combos_sarimax = list(itertools.product(
    p_range, d_range, q_range, P_range, D_range, Q_range, [SEASONAL_PERIOD]
))

n_validacao = min(VALIDATION_HOURS, max(24 * 7, len(y_train) // 10))
y_grid_train, y_validacao = y_train.iloc[:-n_validacao], y_train.iloc[-n_validacao:]
X_grid_train, X_validacao = X_train.iloc[:-n_validacao], X_train.iloc[-n_validacao:]
scaler_y_grid, scaler_x_grid = StandardScaler(), StandardScaler()
y_grid_train_sc = pd.Series(
    scaler_y_grid.fit_transform(y_grid_train.to_numpy().reshape(-1, 1)).ravel(),
    index=y_grid_train.index,
)
X_grid_train_sc = pd.DataFrame(
    scaler_x_grid.fit_transform(X_grid_train), index=X_grid_train.index,
    columns=X_grid_train.columns,
)
X_validacao_sc = pd.DataFrame(
    scaler_x_grid.transform(X_validacao), index=X_validacao.index,
    columns=X_validacao.columns,
)
y_validacao_sc = pd.Series(
    scaler_y_grid.transform(y_validacao.to_numpy().reshape(-1, 1)).ravel(),
    index=y_validacao.index,
)

sarimax_grid = []
for i, (p, d, q, P, D, Q, m) in enumerate(combos_sarimax, start=1):
    if d + D == 0 and p + q + P + Q == 0:
        continue
    try:
        estado = SARIMAX(
            y_grid_train_sc.to_numpy(), exog=X_grid_train_sc.to_numpy(),
            order=(p, d, q), seasonal_order=(P, D, Q, m), trend=TREND,
            enforce_stationarity=ENFORCE_STATIONARITY,
            enforce_invertibility=ENFORCE_INVERTIBILITY,
        ).fit(disp=False, maxiter=100)
        previstos = []
        for data in y_validacao.index:
            x_atual = X_validacao_sc.loc[[data]]
            previstos.append(float(np.asarray(estado.get_forecast(1, exog=x_atual.to_numpy()).predicted_mean)[0]))
            estado = estado.append(
                [y_validacao_sc.loc[data]], exog=x_atual.to_numpy(), refit=False
            )
        previstos = scaler_y_grid.inverse_transform(
            np.asarray(previstos).reshape(-1, 1)
        ).ravel()
        sarimax_grid.append({
            "order": (p, d, q), "seasonal_order": (P, D, Q, m),
            "AIC": estado.aic, "BIC": estado.bic,
            "MAE_validacao": mean_absolute_error(y_validacao, previstos),
        })
    except Exception as exc:
        print(f"Combinação {(p, d, q, P, D, Q, m)} falhou: {type(exc).__name__}")
    if i % 14 == 0:
        print(f"{i}/{len(combos_sarimax)} combinações testadas")

if not sarimax_grid:
    raise RuntimeError("Nenhuma combinação SARIMAX convergiu.")
df_sarimax_grid = pd.DataFrame(sarimax_grid).sort_values(
    ["MAE_validacao", "AIC"]
).reset_index(drop=True)
display(df_sarimax_grid.head(10))
ORDER = tuple(df_sarimax_grid.loc[0, "order"])
SEASONAL_ORDER = tuple(df_sarimax_grid.loc[0, "seasonal_order"])
print(f"Configuração congelada: {ORDER} x {SEASONAL_ORDER}")


# 5. Avaliação

## T16 — Comparação por MAE

Consolidação do MAE fora da amostra por base e modelo, incluindo número de vitórias e posição média.

In [ ]:
# Refit no treino completo e walk-forward de uma hora no teste.
scaler_y, scaler_x = StandardScaler(), StandardScaler()
y_train_scaled = pd.Series(
    scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel(),
    index=y_train.index, name="target",
)
X_train_scaled = pd.DataFrame(
    scaler_x.fit_transform(X_train), index=X_train.index, columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler_x.transform(X_test), index=X_test.index, columns=X_test.columns
)
y_test_scaled = pd.Series(
    scaler_y.transform(y_test.to_numpy().reshape(-1, 1)).ravel(),
    index=y_test.index, name="target",
)

modelo = SARIMAX(
    y_train_scaled.to_numpy(), exog=X_train_scaled.to_numpy(), order=ORDER,
    seasonal_order=SEASONAL_ORDER, trend=TREND,
    enforce_stationarity=ENFORCE_STATIONARITY,
    enforce_invertibility=ENFORCE_INVERTIBILITY,
).fit(disp=False, maxiter=100)
resultado = modelo
estado = modelo
previsoes_scaled = []
for i, data in enumerate(y_test.index, start=1):
    x_atual = X_test_scaled.loc[[data]]
    previsoes_scaled.append(float(
        np.asarray(estado.get_forecast(steps=1, exog=x_atual.to_numpy()).predicted_mean)[0]
    ))
    estado = estado.append([y_test_scaled.loc[data]], exog=x_atual.to_numpy(), refit=False)
    if i % 1000 == 0:
        print(f"{i}/{len(y_test)} previsões")

previsao = scaler_y.inverse_transform(
    np.asarray(previsoes_scaled).reshape(-1, 1)
).ravel()
erros = y_test.to_numpy() - previsao
metricas = pd.DataFrame([{
    "base": "clima", "modelo": f"SARIMAX {ORDER}{SEASONAL_ORDER}",
    "AIC": modelo.aic, "BIC": modelo.bic,
    "MAE": mean_absolute_error(y_test, previsao),
    "RMSE": mean_squared_error(y_test, previsao) ** 0.5,
    "ME_vies": erros.mean(), "horizonte_horas": HORIZON,
}])
display(metricas.round(4))


## T17 — Diagnóstico dos resíduos

Análise gráfica e estatística dos resíduos com série temporal dos erros, ACF, teste de Ljung-Box e tabela de p-valores.

In [ ]:
# Diagnóstico dos resíduos fora da amostra
residuos_teste = pd.Series(erros, index=y_test.index, name="residuo")
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
axes[0].plot(
    residuos_teste.index,
    residuos_teste,
    color="#1a3a6e",
    label="Erro de previsão",
)
axes[0].axhline(0, color="#e25c3e", linestyle="--", label="Erro zero")
axes[0].set_title("Resíduos fora da amostra — SARIMAX")
axes[0].set_ylabel("Erro (°C)")
axes[0].set_xlabel("Data e hora")
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

plot_acf(
    residuos_teste,
    lags=48,
    ax=axes[1],
    title="ACF dos resíduos fora da amostra",
)
axes[1].set_xlabel("Defasagem (lag)")
axes[1].set_ylabel("Autocorrelação")
axes[1].legend(["Resíduos do teste"], loc="upper right")
axes[1].grid(True, alpha=0.3)

fig.suptitle("Diagnóstico dos resíduos — grupo 4", fontsize=14)
plt.tight_layout()
plt.show()
analise_residuos_teste = analisar_componente_residual(
    type("Decomposicao", (), {"resid": residuos_teste})(), lags=24
)
display(analise_residuos_teste["autocorrelacao_residual"])
print(f"Viés médio: {residuos_teste.mean():.4f}")

## T18 — Importância das features

Cálculo e interpretação da importância das features: permutation importance para modelos de tabela e coeficientes para o SARIMAX.

In [ ]:
# Como X e y foram padronizados, os módulos dos coeficientes das exógenas
# são comparáveis entre si. Termos AR, MA e sazonais não entram na tabela.
importancias_sarimax = calcular_importancia_features(
    resultado, X_test_scaled, metodo="coeficientes"
)
display(importancias_sarimax.round(4))
top = importancias_sarimax.head(15).sort_values("importancia")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["feature"], top["importancia"])
ax.set_title("T18 — Importância das exógenas do SARIMAX")
ax.set_xlabel("Módulo do coeficiente padronizado")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
out_t18 = PROJECT_ROOT / "outputs" / "t18" / "grupo4"
out_t18.mkdir(parents=True, exist_ok=True)
fig.savefig(out_t18 / "importancia_coeficientes_sarimax.png", dpi=150, bbox_inches="tight")
plt.show()
importancias_sarimax.to_csv(out_t18 / "importancia_coeficientes_sarimax.csv", index=False)